# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [2]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [3]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 74s 0us/step
Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [4]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y

        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [5]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [7]:
device = '/device:GPU:0'
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [8]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.flatten = tf.keras.layers.Flatten()

        self.fc = tf.keras.layers.Dense(
            units=num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores

In [9]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [14]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False, print_every=100):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):


        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()

        model = model_init_fn()
        optimizer = optimizer_init_fn()

        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

        t = 0
        for epoch in range(num_epochs):

            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()

            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:

                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)

                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)

                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)

                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [15]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.0210869312286377, Accuracy: 6.25, Val Loss: 3.083047389984131, Val Accuracy: 9.199999809265137
Iteration 100, Epoch 1, Loss: 2.276326894760132, Accuracy: 27.831066131591797, Val Loss: 1.9516535997390747, Val Accuracy: 38.0
Iteration 200, Epoch 1, Loss: 2.0962071418762207, Accuracy: 32.019588470458984, Val Loss: 1.8418389558792114, Val Accuracy: 40.099998474121094
Iteration 300, Epoch 1, Loss: 2.012030601501465, Accuracy: 34.022010803222656, Val Loss: 1.8628873825073242, Val Accuracy: 36.099998474121094
Iteration 400, Epoch 1, Loss: 1.9404616355895996, Accuracy: 35.85956954956055, Val Loss: 1.7811726331710815, Val Accuracy: 41.29999923706055
Iteration 500, Epoch 1, Loss: 1.8926045894622803, Accuracy: 37.041542053222656, Val Loss: 1.7024285793304443, Val Accuracy: 43.20000076293945
Iteration 600, Epoch 1, Loss: 1.8611385822296143, Accuracy: 37.973167419433594, Val Loss: 1.7028985023498535, Val Accuracy: 41.900001525878906
Iteration 700, Epoch 1, Loss: 1.8352

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [16]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.2524919509887695, Accuracy: 9.375, Val Loss: 4.325611591339111, Val Accuracy: 8.5
Iteration 100, Epoch 1, Loss: 1.96029794216156, Accuracy: 31.141706466674805, Val Loss: 1.6826426982879639, Val Accuracy: 41.60000228881836
Iteration 200, Epoch 1, Loss: 1.791530728340149, Accuracy: 37.21237564086914, Val Loss: 1.5444741249084473, Val Accuracy: 46.0
Iteration 300, Epoch 1, Loss: 1.701897382736206, Accuracy: 40.36025619506836, Val Loss: 1.5461394786834717, Val Accuracy: 46.70000076293945
Iteration 400, Epoch 1, Loss: 1.6250061988830566, Accuracy: 42.81093978881836, Val Loss: 1.3637713193893433, Val Accuracy: 50.900001525878906
Iteration 500, Epoch 1, Loss: 1.5687434673309326, Accuracy: 44.59830093383789, Val Loss: 1.3295999765396118, Val Accuracy: 52.20000076293945
Iteration 600, Epoch 1, Loss: 1.5349767208099365, Accuracy: 45.65047836303711, Val Loss: 1.2885332107543945, Val Accuracy: 56.19999694824219
Iteration 700, Epoch 1, Loss: 1.5033375024795532, Accurac

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [17]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.035954475402832, Accuracy: 4.6875, Val Loss: 3.020928144454956, Val Accuracy: 10.899999618530273
Iteration 100, Epoch 1, Loss: 2.2508022785186768, Accuracy: 28.496286392211914, Val Loss: 1.9123921394348145, Val Accuracy: 37.70000076293945
Iteration 200, Epoch 1, Loss: 2.0877044200897217, Accuracy: 32.06623077392578, Val Loss: 1.9107179641723633, Val Accuracy: 39.70000076293945
Iteration 300, Epoch 1, Loss: 2.0082309246063232, Accuracy: 34.04796600341797, Val Loss: 1.8862497806549072, Val Accuracy: 38.0
Iteration 400, Epoch 1, Loss: 1.93844473361969, Accuracy: 35.89853286743164, Val Loss: 1.7054674625396729, Val Accuracy: 42.599998474121094
Iteration 500, Epoch 1, Loss: 1.8929132223129272, Accuracy: 37.02906799316406, Val Loss: 1.6663529872894287, Val Accuracy: 43.20000076293945
Iteration 600, Epoch 1, Loss: 1.8621211051940918, Accuracy: 38.00956726074219, Val Loss: 1.6756477355957031, Val Accuracy: 43.20000076293945
Iteration 700, Epoch 1, Loss: 1.83602678

Альтернативный менее гибкий способ обучения:

In [18]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.8241 - sparse_categorical_accuracy: 0.3855 - val_loss: 1.6731 - val_sparse_categorical_accuracy: 0.4160
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.6573 - sparse_categorical_accuracy: 0.4223


[1.657339096069336, 0.4223000109195709]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [19]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    input_shape = (32, 32, 3)
    channel_1, channel_2, num_classes = 32, 16, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=5,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        ),

        tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=3,
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        ),

        tf.keras.layers.Flatten(),

        tf.keras.layers.Dense(
            units=num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.6499624252319336, Accuracy: 12.5, Val Loss: 2.5624594688415527, Val Accuracy: 10.300000190734863
Iteration 100, Epoch 1, Loss: 2.026744842529297, Accuracy: 28.001235961914062, Val Loss: 1.8576672077178955, Val Accuracy: 36.19999694824219
Iteration 200, Epoch 1, Loss: 1.9157136678695679, Accuracy: 32.563743591308594, Val Loss: 1.7402034997940063, Val Accuracy: 41.400001525878906
Iteration 300, Epoch 1, Loss: 1.8481652736663818, Accuracy: 34.70722579956055, Val Loss: 1.6902114152908325, Val Accuracy: 41.900001525878906
Iteration 400, Epoch 1, Loss: 1.7849611043930054, Accuracy: 37.06748962402344, Val Loss: 1.6264313459396362, Val Accuracy: 44.0
Iteration 500, Epoch 1, Loss: 1.7394299507141113, Accuracy: 38.64458465576172, Val Loss: 1.564129114151001, Val Accuracy: 46.29999923706055
Iteration 600, Epoch 1, Loss: 1.708548903465271, Accuracy: 39.89704513549805, Val Loss: 1.5285863876342773, Val Accuracy: 46.599998474121094
Iteration 700, Epoch 1, Loss: 1.680057

In [20]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.6241 - sparse_categorical_accuracy: 0.4314 - val_loss: 1.4754 - val_sparse_categorical_accuracy: 0.4920
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.4604 - sparse_categorical_accuracy: 0.4815


[1.4604443311691284, 0.4814999997615814]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [21]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [22]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.2512617111206055, Accuracy: 4.6875, Val Loss: 2.9414849281311035, Val Accuracy: 12.700000762939453
Iteration 100, Epoch 1, Loss: 2.235492706298828, Accuracy: 28.681930541992188, Val Loss: 1.8999334573745728, Val Accuracy: 38.60000228881836
Iteration 200, Epoch 1, Loss: 2.078852653503418, Accuracy: 32.190608978271484, Val Loss: 1.8876469135284424, Val Accuracy: 39.099998474121094
Iteration 300, Epoch 1, Loss: 2.0077335834503174, Accuracy: 34.01681900024414, Val Loss: 1.8829612731933594, Val Accuracy: 36.5
Iteration 400, Epoch 1, Loss: 1.9392690658569336, Accuracy: 35.82839584350586, Val Loss: 1.7171661853790283, Val Accuracy: 42.89999771118164
Iteration 500, Epoch 1, Loss: 1.8947268724441528, Accuracy: 36.8357048034668, Val Loss: 1.650781512260437, Val Accuracy: 44.29999923706055
Iteration 600, Epoch 1, Loss: 1.8637897968292236, Accuracy: 37.68718719482422, Val Loss: 1.6890031099319458, Val Accuracy: 42.89999771118164
Iteration 700, Epoch 1, Loss: 1.8373739

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [23]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding='same', kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(2)

        self.conv2 = tf.keras.layers.Conv2D(64, 3, padding='same', kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(2)

        self.conv3 = tf.keras.layers.Conv2D(128, 3, padding='same', kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.pool3 = tf.keras.layers.MaxPooling2D(2)

        self.conv4 = tf.keras.layers.Conv2D(256, 3, padding='same', kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()

        self.global_pool = tf.keras.layers.GlobalAveragePooling2D()
        self.dropout1 = tf.keras.layers.Dropout(0.5)
        self.fc1 = tf.keras.layers.Dense(256, activation='relu', kernel_initializer=initializer)
        self.dropout2 = tf.keras.layers.Dropout(0.3)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        x = input_tensor

        x = self.conv1(x)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool3(x)

        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)

        x = self.global_pool(x)
        x = self.dropout1(x, training=training)
        x = self.fc1(x)
        x = self.dropout2(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 2.940596580505371, Accuracy: 10.9375, Val Loss: 3.1968204975128174, Val Accuracy: 11.200000762939453
Iteration 100, Epoch 1, Loss: 1.9507869482040405, Accuracy: 29.6875, Val Loss: 1.8637726306915283, Val Accuracy: 36.29999923706055
Iteration 200, Epoch 1, Loss: 1.7961090803146362, Accuracy: 34.49160385131836, Val Loss: 1.6490403413772583, Val Accuracy: 40.20000076293945
Iteration 300, Epoch 1, Loss: 1.7049635648727417, Accuracy: 37.67649459838867, Val Loss: 1.8158528804779053, Val Accuracy: 34.0
Iteration 400, Epoch 1, Loss: 1.6290680170059204, Accuracy: 40.46134567260742, Val Loss: 1.7410510778427124, Val Accuracy: 38.70000076293945
Iteration 500, Epoch 1, Loss: 1.5716959238052368, Accuracy: 42.54303741455078, Val Loss: 1.4545286893844604, Val Accuracy: 46.599998474121094
Iteration 600, Epoch 1, Loss: 1.528814435005188, Accuracy: 44.08797836303711, Val Loss: 1.727838158607483, Val Accuracy: 39.099998474121094
Iteration 700, Epoch 1, Loss: 1.4934954643249512

In [24]:
class CustomConvNet_2(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet_2, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu', kernel_initializer=initializer)
        self.pool1 = tf.keras.layers.MaxPooling2D(2)

        self.conv2 = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu', kernel_initializer=initializer)
        self.pool2 = tf.keras.layers.MaxPooling2D(2)

        self.conv3 = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu', kernel_initializer=initializer)
        self.pool3 = tf.keras.layers.MaxPooling2D(2)

        self.flatten = tf.keras.layers.Flatten()
        self.dropout1 = tf.keras.layers.Dropout(0.5)
        self.fc1 = tf.keras.layers.Dense(512, activation='relu', kernel_initializer=initializer)
        self.dropout2 = tf.keras.layers.Dropout(0.3)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        x = input_tensor

        x = self.conv1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.pool3(x)

        x = self.flatten(x)
        x = self.dropout1(x, training=training)
        x = self.fc1(x)
        x = self.dropout2(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet_2()

def model_init_fn():
    return CustomConvNet_2()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate)

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 6.121238708496094, Accuracy: 12.5, Val Loss: 16.0456600189209, Val Accuracy: 7.90000057220459
Iteration 100, Epoch 1, Loss: 2.3981170654296875, Accuracy: 27.753713607788086, Val Loss: 1.6683226823806763, Val Accuracy: 40.5
Iteration 200, Epoch 1, Loss: 2.058361291885376, Accuracy: 32.72698974609375, Val Loss: 1.4778540134429932, Val Accuracy: 50.0
Iteration 300, Epoch 1, Loss: 1.899353265762329, Accuracy: 36.1451416015625, Val Loss: 1.4171626567840576, Val Accuracy: 48.400001525878906
Iteration 400, Epoch 1, Loss: 1.7934565544128418, Accuracy: 38.614402770996094, Val Loss: 1.3276197910308838, Val Accuracy: 54.70000076293945
Iteration 500, Epoch 1, Loss: 1.7155202627182007, Accuracy: 40.74974822998047, Val Loss: 1.30288565158844, Val Accuracy: 54.599998474121094
Iteration 600, Epoch 1, Loss: 1.661993384361267, Accuracy: 42.20829772949219, Val Loss: 1.2275261878967285, Val Accuracy: 59.20000076293945
Iteration 700, Epoch 1, Loss: 1.6138081550598145, Accuracy: 

Опишите все эксперименты, результаты. Сделайте выводы.

**Эксперимент 1**

Исследование глубокой свёрточной сети с Batch Normalization и Dropout для классификации изображений CIFAR-10

Результаты:

* максимальная точность на валидации — 76,8%
* точность на валидации к концу 10-й эпохи составила 71,3%, что немного ниже максимальной
* точность на обучающей выборке выросла до 83,7% к концу 10 эпохи, максимальная точность составила 85,7%
* финальный loss на обучении: 0,474
* финальный loss на валидации: 0,933

Данная архитектура демонстрирует отсутствие явного переобучения. Точность на валидации стабильно росла на протяжении всех 10 эпох и практически не снизилась к концу обучения, что свидетельствует о хорошей способности к обобщению.

**Эксперимент 2**

Исследование глубокой свёрточной сети без Batch Normalization, но с увеличенным количеством фильтров и более мощным классификатором для классификации изображений CIFAR-10

Результаты:

* максимальная точность на валидации — 80,1% (достигнута к 10-й эпохе)
* точность на обучающей выборке выросла до 82,0% к концу 10 эпохи, максимальная точность составила 83,7%
* финальный loss на обучении: 0,511
* финальный loss на валидации: 0,635

Данная архитектура показала результаты, превосходящие первый эксперимент с Batch Normalization. Несмотря на отсутствие BN, сеть достигла 80,1% точности на валидации, что выше, чем в эксперименте 1. Ключевыми факторами успеха стали увеличенное количество фильтров в начальных слоях (64 вместо 32), более мощный классификатор и использование Flatten вместо GlobalAveragePooling, что позволило сохранить больше пространственной информации. Разрыв между точностью на обучающей выборке и валидационной очень маленький, что говорит об отсутствии переобучения.

Увеличение ёмкости модели в сочетании с Dropout может быть более эффективным для CIFAR-10, чем добавление Batch Normalization при сохранении умеренной архитектуры.